# Ishan Bashyal 240530/15939241

### Part A: Data Loading and Inspection

Q1. Load the supermarket sales dataset into a PySpark DataFrame.

In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("EDA").master("local[*]").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/07 12:23:16 WARN Utils: Your hostname, Inishs-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 10.1.21.202 instead (on interface en0)
26/06/07 12:23:16 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/07 12:23:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/07 12:23:17 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Q2. Display: - First 10 records - Schema of the dataset - Total number of rows and columns

In [2]:
df = spark.read.csv("supermarket_sales.csv", header=True, inferSchema=True)
df.printSchema()
df.show(10)

root
 |-- transaction_id: string (nullable = true)
 |-- date: date (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- total_sales: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)

+--------------+----------+-----------+----------------+------------+--------+----------+-----------+--------------+----------+
|transaction_id|      date|customer_id|product_category|product_name|quantity|unit_price|total_sales|payment_method|      city|
+--------------+----------+-----------+----------------+------------+--------+----------+-----------+--------------+----------+
|         T0001|2025-02-05|       C027|       Beverages|      Coffee|       5|     13.38|       66.9|       eWallet|Biratnagar|
|         T0002|2025-02-25|       C060|       Beverages|  S

In [3]:
print("Row count   :", df.count())
print("Column count:", len(df.columns))

Row count   : 500
Column count: 10


Q3. Check for: - Null values - Duplicate records

In [4]:
from pyspark.sql.functions import col, sum as spark_sum, when

null_count = df.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
])

null_count.show()

+--------------+----+-----------+----------------+------------+--------+----------+-----------+--------------+----+
|transaction_id|date|customer_id|product_category|product_name|quantity|unit_price|total_sales|payment_method|city|
+--------------+----+-----------+----------------+------------+--------+----------+-----------+--------------+----+
|             0|   0|          0|               0|           0|       0|         0|          0|             0|   0|
+--------------+----+-----------+----------------+------------+--------+----------+-----------+--------------+----+



In [5]:
duplicate_count = df.groupBy(df.columns).count().filter(col("count") > 1)


In [6]:
duplicate_count.show()

+--------------+----+-----------+----------------+------------+--------+----------+-----------+--------------+----+-----+
|transaction_id|date|customer_id|product_category|product_name|quantity|unit_price|total_sales|payment_method|city|count|
+--------------+----+-----------+----------------+------------+--------+----------+-----------+--------------+----+-----+
+--------------+----+-----------+----------------+------------+--------+----------+-----------+--------------+----+-----+



Q4. Generate descriptive statistics for all numerical columns.

In [7]:
df.describe().show()

26/06/07 12:23:21 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+--------------+-----------+----------------+------------+------------------+------------------+-----------------+--------------+---------+
|summary|transaction_id|customer_id|product_category|product_name|          quantity|        unit_price|      total_sales|payment_method|     city|
+-------+--------------+-----------+----------------+------------+------------------+------------------+-----------------+--------------+---------+
|  count|           500|        500|             500|         500|               500|               500|              500|           500|      500|
|   mean|          NULL|       NULL|            NULL|        NULL|             5.524|25.628980000000013|        145.03284|          NULL|     NULL|
| stddev|          NULL|       NULL|            NULL|        NULL|2.9730627649628962|13.991613046066618|120.2808106872146|          NULL|     NULL|
|    min|         T0001|       C002|          Bakery|       Apple|                 1|              1.73|        

### Part B: EDA Using PySpark DataFrame Methods

Q5. Calculate the total supermarket revenue.

In [8]:
from pyspark.sql.functions import sum

df.select(sum("total_sales")).show()

+----------------+
|sum(total_sales)|
+----------------+
|        72516.42|
+----------------+



Q6. Find the top 5 highest-selling product categories based on total sales.

In [9]:
df.groupBy("product_category").sum("total_sales").orderBy(col("sum(total_sales)").desc()).show(5)

+----------------+------------------+
|product_category|  sum(total_sales)|
+----------------+------------------+
|          Snacks|15938.829999999998|
|           Dairy|15883.500000000004|
|          Bakery|14209.069999999994|
|       Beverages|14186.970000000005|
|         Produce|12298.050000000001|
+----------------+------------------+



Q7. Determine: - Average quantity sold - Average unit price - Average sales amount

In [10]:
from pyspark.sql.functions import avg

df.select(avg("quantity"), avg("unit_price"), avg("total_sales")).show()

+-------------+------------------+----------------+
|avg(quantity)|   avg(unit_price)|avg(total_sales)|
+-------------+------------------+----------------+
|        5.524|25.628980000000013|       145.03284|
+-------------+------------------+----------------+



Q8. Find the city generating the highest revenue.

In [11]:
df.groupBy("city").sum("total_sales").orderBy(col("sum(total_sales)").desc()).show(1)

+---------+------------------+
|     city|  sum(total_sales)|
+---------+------------------+
|Bhaktapur|16650.069999999996|
+---------+------------------+
only showing top 1 row


Q9. Identify the payment method used most frequently.

In [12]:
df.groupBy("payment_method").count().orderBy(col("count").desc()).show()

+--------------+-----+
|payment_method|count|
+--------------+-----+
|          Card|  177|
|          Cash|  163|
|       eWallet|  160|
+--------------+-----+



Q10. Calculate total sales for each product category.

In [13]:
df.groupBy("product_category").sum("total_sales").show()

+----------------+------------------+
|product_category|  sum(total_sales)|
+----------------+------------------+
|          Bakery|14209.069999999994|
|       Beverages|14186.970000000005|
|           Dairy|15883.500000000004|
|         Produce|12298.050000000001|
|          Snacks|15938.829999999998|
+----------------+------------------+



Q11. Find the top 10 transactions with the highest sales amount.

In [14]:
df.orderBy(col("total_sales").desc()).show(10)

+--------------+----------+-----------+----------------+------------+--------+----------+-----------+--------------+----------+
|transaction_id|      date|customer_id|product_category|product_name|quantity|unit_price|total_sales|payment_method|      city|
+--------------+----------+-----------+----------------+------------+--------+----------+-----------+--------------+----------+
|         T0015|2025-05-16|       C065|          Snacks|    Crackers|      10|     49.81|      498.1|       eWallet| Kathmandu|
|         T0093|2025-05-15|       C073|          Snacks|   Chocolate|      10|     48.87|      488.7|          Cash|   Pokhara|
|         T0100|2025-01-26|       C054|          Bakery|       Bread|      10|     48.07|      480.7|       eWallet|   Pokhara|
|         T0161|2025-05-24|       C085|           Dairy|        Milk|      10|      47.9|      479.0|       eWallet| Kathmandu|
|         T0098|2025-01-22|       C064|           Dairy|      Cheese|      10|     47.53|      475.3|   

Q12. Create a new column named sales_level:
High → sales > 100 - Medium → sales between 50 and 100 - Low → sales < 50
Count the number of transactions in each sales level.

In [15]:
df2 = df.withColumn("sales_level", 
    when(col("total_sales") > 100, "High")
    .when((col("total_sales") >= 50) & (col("total_sales") <= 100), "Medium")
    .otherwise("Low")
)

In [16]:
df2.groupBy("sales_level").count().show()

+-----------+-----+
|sales_level|count|
+-----------+-----+
|       High|  261|
|        Low|  139|
|     Medium|  100|
+-----------+-----+



### Part C: EDA Using Spark SQL

Q13. Create a temporary SQL view named:

In [17]:
df.createOrReplaceTempView("sales_view")

Q14. Using Spark SQL, write queries to answer the following:
A. Total revenue generated by the supermarket.

In [18]:
spark.sql("""
SELECT SUM(total_sales) AS total_revenue
FROM sales_view
""").show()

+-------------+
|total_revenue|
+-------------+
|     72516.42|
+-------------+



B. Revenue generated by each city.

In [19]:
spark.sql("""
SELECT city, SUM(total_sales) AS revenue
FROM sales_view
GROUP BY city
""").show()

+----------+------------------+
|      city|           revenue|
+----------+------------------+
|   Pokhara|14540.320000000003|
|  Lalitpur|16323.509999999998|
| Kathmandu|          13034.83|
| Bhaktapur|16650.069999999996|
|Biratnagar|11967.690000000002|
+----------+------------------+



C. Revenue generated by each product category.

In [20]:
spark.sql("""
SELECT product_category, SUM(total_sales) AS revenue
FROM sales_view
GROUP BY product_category
""").show()

+----------------+------------------+
|product_category|           revenue|
+----------------+------------------+
|          Bakery|14209.069999999994|
|       Beverages|14186.970000000005|
|           Dairy|15883.500000000004|
|         Produce|12298.050000000001|
|          Snacks|15938.829999999998|
+----------------+------------------+



D. Average quantity sold for each category.

In [21]:
spark.sql("""
SELECT product_category, AVG(quantity) AS avg_quantity
FROM sales_view
GROUP BY product_category
""").show()

+----------------+------------------+
|product_category|      avg_quantity|
+----------------+------------------+
|          Bakery|5.6020408163265305|
|       Beverages| 5.245283018867925|
|           Dairy| 5.904255319148936|
|         Produce| 5.540816326530612|
|          Snacks|             5.375|
+----------------+------------------+



E. Top 5 products by total sales.

In [22]:
spark.sql("""
SELECT product_name, SUM(total_sales) AS total_sales
FROM sales_view
GROUP BY product_name
ORDER BY total_sales DESC
LIMIT 5
""").show()

+------------+------------------+
|product_name|       total_sales|
+------------+------------------+
|         Tea| 5265.120000000001|
|        Milk| 5056.429999999999|
|     Cookies| 4913.889999999999|
|       Apple|4596.6100000000015|
|      Yogurt| 4299.000000000001|
+------------+------------------+



F. Number of transactions for each payment method.

In [23]:
spark.sql("""
SELECT payment_method, COUNT(*) AS transactions
FROM sales_view
GROUP BY payment_method
""").show()

+--------------+------------+
|payment_method|transactions|
+--------------+------------+
|          Card|         177|
|          Cash|         163|
|       eWallet|         160|
+--------------+------------+



G. Highest single transaction amount.

In [24]:
spark.sql("""
SELECT MAX(total_sales) AS highest_transaction
FROM sales_view
""").show()

+-------------------+
|highest_transaction|
+-------------------+
|              498.1|
+-------------------+



H. Daily sales summary: - Date - Total Revenue - Number of Transactions

In [25]:
spark.sql("""
SELECT date,
       SUM(total_sales) AS total_revenue,
       COUNT(*) AS transactions
FROM sales_view
GROUP BY date
ORDER BY date
""").show()

+----------+------------------+------------+
|      date|     total_revenue|transactions|
+----------+------------------+------------+
|2025-01-01|185.85000000000002|           2|
|2025-01-02|             223.3|           1|
|2025-01-03|            111.04|           1|
|2025-01-04|            688.55|           4|
|2025-01-05|             160.4|           1|
|2025-01-07|            132.75|           1|
|2025-01-08|            433.02|           2|
|2025-01-09|            828.78|           4|
|2025-01-10|392.63000000000005|           3|
|2025-01-11|              43.2|           1|
|2025-01-12|             526.3|           3|
|2025-01-13|            621.75|           3|
|2025-01-14| 897.1800000000001|           4|
|2025-01-15|            945.99|           5|
|2025-01-16|            658.35|           2|
|2025-01-17|            229.68|           2|
|2025-01-18| 852.9499999999999|           5|
|2025-01-19|            288.01|           3|
|2025-01-20|            580.51|           5|
|2025-01-2

### Part D: Insights and Interpretation

##### Bhaktapur and Lalitpur generate the most revenue for the supermarket, while Biratnagar brings in the least.  
##### More than half of the store's total sales come from high-value transactions that exceed $100.  
##### Snacks and Dairy are the top-selling food categories, but Tea is the highest-selling individual product.  
##### Customers prefer digital payments, with Cards and eWallets heavily outperforming traditional Cash.  
##### Dairy products are bought in the highest quantities, averaging nearly 6 items per shopping basket.